Install and import packages

In [1]:
# Install required packages
!pip -q install datasets pandas numpy scikit-learn



In [2]:
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

Load the DGEB EC classification dataset

In [3]:
# Load dataset from Hugging Face
dataset = load_dataset("tattabio/ec_classification")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/485 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/285k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/87.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/512 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Entry', 'Label', 'Sequence'],
        num_rows: 512
    })
    test: Dataset({
        features: ['Entry', 'Label', 'Sequence'],
        num_rows: 128
    })
})


In [4]:
# Convert train and test splits to pandas DataFrames
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (512, 3)
Test shape: (128, 3)


,Entry,Label,Sequence
0,Q9LQC0,1.14.14.18,MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHL...
1,A0AFT8,1.14.14.18,MIIVTNTIKVEKGAAEHVIRQFTGANGDGHPTKDIAEVEGFLGFEL...
2,P74133,1.14.14.18,MTNLAQKLRYGTQQSHTLAENTAYMKCFLKGIVEREPFRQLLANLY...
3,O31534,1.14.14.18,MFVQLRKMTVKEGFADKVIERFSAEGIIEKQEGLIDVTVLEKNVRR...
4,P34697,1.15.1.1,MFMNLLTQVSNAIFPQVEAAQKMSNRAVAVLRGETVTGTIWITQKS...


In [5]:
print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

Train columns: ['Entry', 'Label', 'Sequence']
Test columns: ['Entry', 'Label', 'Sequence']


In [6]:
print("Example training row:")
print(train_df.iloc[0])

Example training row:
Entry                                                  Q9LQC0
Label                                              1.14.14.18
Sequence    MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHL...
Name: 0, dtype: object


In [7]:
print("Number of unique labels in train:", train_df["Label"].nunique())
print("Number of unique labels in test:", test_df["Label"].nunique())

Number of unique labels in train: 128
Number of unique labels in test: 128


In [8]:
print("Top 10 label counts in training set:")
print(train_df["Label"].value_counts().head(10))

Top 10 label counts in training set:
Label
1.14.14.18    4
1.15.1.1      4
1.16.3.1      4
1.17.4.1      4
1.5.98.3      4
1.8.3.2       4
2.1.1.354     4
2.1.1.360     4
2.1.1.37      4
2.1.1.72      4
Name: count, dtype: int64


Define amino acid frequency feature extraction

Each protein sequence will be converted into a 20-dimensional feature vector

In [9]:
# 20 standard amino acids
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

print("Amino acids:", AMINO_ACIDS)

Amino acids: ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']


In [10]:
def sequence_to_aa_frequency(seq):
    """
    Inputs
    ----------
    seq : str
        Protein sequence.

    Returns
    -------
    np.ndarray
        A normalized frequency vector of length 20.
    """
    seq = str(seq).upper()
    counts = np.zeros(len(AMINO_ACIDS), dtype=float)
    valid_count = 0

    for ch in seq:
        if ch in AA_TO_INDEX:
            counts[AA_TO_INDEX[ch]] += 1
            valid_count += 1

    if valid_count > 0:
        counts /= valid_count

    return counts

In [11]:
def build_feature_matrix(sequences):
    """
    Inputs
    ----------
    sequences : iterable
        Protein sequences.

    Returns
    -------
    np.ndarray
        Shape (n_samples, 20)
    """
    return np.array([sequence_to_aa_frequency(seq) for seq in sequences])

Build X and y

X_train, X_test: amino acid frequency features


y_train, y_test: encoded EC labels

In [12]:
# Build feature matrices
X_train = build_feature_matrix(train_df["Sequence"])
X_test = build_feature_matrix(test_df["Sequence"])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (512, 20)
X_test shape: (128, 20)


In [13]:
# Encode string labels into integers
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["Label"])
y_test = label_encoder.transform(test_df["Label"])

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("Number of classes:", len(label_encoder.classes_))

y_train shape: (512,)
y_test shape: (128,)
Number of classes: 128


In [14]:
print("First 10 label classes:")
print(label_encoder.classes_[:10])

First 10 label classes:
['1.14.14.18' '1.15.1.1' '1.16.3.1' '1.17.4.1' '1.5.98.3' '1.8.3.2'
 '2.1.1.354' '2.1.1.360' '2.1.1.37' '2.1.1.72']


Quick sanity check

In [15]:
# Show the amino acid frequency vector for the first training sequence
example_seq = train_df.loc[0, "Sequence"]
example_vec = sequence_to_aa_frequency(example_seq)

print("Example sequence:")
print(example_seq[:120] + "..." if len(example_seq) > 120 else example_seq)
print("\nFeature vector:")
print(example_vec)
print("\nSum of frequencies:", example_vec.sum())

Example sequence:
MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHLVRRANEDQTLVVNVVAAAGEKPERRYPREPNGFVEEMRFVVMKIHPRDQVKEGKSDSNDLVSTWNFTIEGYLKF...

Feature vector:
[0.07420495 0.01413428 0.03533569 0.1130742  0.04946996 0.04240283
 0.01766784 0.04946996 0.07067138 0.08833922 0.01766784 0.0565371
 0.03533569 0.02473498 0.07773852 0.06360424 0.04240283 0.06713781
 0.01766784 0.04240283]

Sum of frequencies: 1.0


In [16]:
# check a few examples in a readable dataframe
feature_preview = pd.DataFrame(
    X_train[:5],
    columns=[f"freq_{aa}" for aa in AMINO_ACIDS]
)
feature_preview["Label"] = train_df["Label"].iloc[:5].values
feature_preview

,freq_A,freq_C,freq_D,freq_E,freq_F,freq_G,freq_H,freq_I,freq_K,freq_L,...,freq_N,freq_P,freq_Q,freq_R,freq_S,freq_T,freq_V,freq_W,freq_Y,Label
0,0.074205,0.014134,0.035336,0.113074,0.049470,0.042403,0.017668,0.049470,0.070671,0.088339,...,0.056537,0.035336,0.024735,0.077739,0.063604,0.042403,0.067138,0.017668,0.042403,1.14.14.18
1,0.057851,0.000000,0.057851,0.132231,0.041322,0.074380,0.041322,0.066116,0.099174,0.016529,...,0.041322,0.024793,0.033058,0.057851,0.041322,0.066116,0.107438,0.024793,0.008264,1.14.14.18
2,0.088000,0.012000,0.052000,0.084000,0.028000,0.052000,0.024000,0.068000,0.036000,0.124000,...,0.044000,0.048000,0.048000,0.068000,0.048000,0.068000,0.024000,0.004000,0.060000,1.14.14.18
3,0.055556,0.000000,0.055556,0.111111,0.027778,0.064815,0.037037,0.064815,0.111111,0.037037,...,0.027778,0.018519,0.046296,0.055556,0.046296,0.037037,0.111111,0.027778,0.027778,1.14.14.18
4,0.105556,0.011111,0.044444,0.055556,0.027778,0.138889,0.038889,0.050000,0.055556,0.061111,...,0.050000,0.038889,0.044444,0.027778,0.044444,0.072222,0.094444,0.005556,0.011111,1.15.1.1


train the model

In [17]:
model = LogisticRegression(
    C=1.0,            # inverse regularization strength
    max_iter=3000,    # maximum iterations
    solver="lbfgs",   # standard solver
    multi_class="auto"
)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(max_iter=3000, multi_class='auto')

In [18]:
y_pred = model.predict(X_test)

evaluate the model

In [19]:
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

Accuracy: 0.0938
Macro F1: 0.0605
Weighted F1: 0.0605


In [20]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Classification Report:
              precision    recall  f1-score   support

  1.14.14.18       0.00      0.00      0.00         1
    1.15.1.1       0.00      0.00      0.00         1
    1.16.3.1       0.00      0.00      0.00         1
    1.17.4.1       0.00      0.00      0.00         1
    1.5.98.3       0.00      0.00      0.00         1
     1.8.3.2       0.00      0.00      0.00         1
   2.1.1.354       0.00      0.00      0.00         1
   2.1.1.360       0.00      0.00      0.00         1
    2.1.1.37       0.00      0.00      0.00         1
    2.1.1.72       0.00      0.00      0.00         1
    2.1.1.77       0.17      1.00      0.29         1
    2.1.1.86       0.33      1.00      0.50         1
    2.1.3.15       0.00      0.00      0.00         1
   2.3.1.225       0.00      0.00      0.00         1
   2.3.1.269       0.20      1.00      0.33         1
    2.3.1.48       0.00      0.00      0.00         1
    2.3.1.51       0.00      0.00      0.00         1
    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


show prediction

In [21]:
pred_labels = label_encoder.inverse_transform(y_pred)
true_labels = label_encoder.inverse_transform(y_test)

results_df = pd.DataFrame({
    "Entry": test_df["Entry"],
    "True_Label": true_labels,
    "Predicted_Label": pred_labels,
    "Correct": true_labels == pred_labels
})

results_df.head(20)

,Entry,True_Label,Predicted_Label,Correct
0,A0A0H2XEA6,1.14.14.18,5.4.99.5,False
1,Q5AD07,1.15.1.1,3.2.1.4,False
2,O74831,1.16.3.1,6.3.4.19,False
3,O46310,1.17.4.1,3.1.11.6,False
4,Q8PU58,1.5.98.3,7.1.1.2,False
5,Q5UQV6,1.8.3.2,7.1.1.2,False
6,Q8IRW8,2.1.1.354,3.2.1.4,False
7,Q6BTC8,2.1.1.360,6.3.4.19,False
8,P0DOY1,2.1.1.37,2.3.2.31,False
9,Q09956,2.1.1.72,3.1.21.4,False


In [22]:
print("Number correct:", results_df["Correct"].sum())
print("Number incorrect:", (~results_df["Correct"]).sum())

Number correct: 12
Number incorrect: 116


Find the best hyperparameters

In [23]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

results = []

for C in [0.01, 0.1, 1.0, 10.0]:
    for max_iter in [500, 1000, 3000]:
        for class_weight in [None, "balanced"]:
            model = LogisticRegression(
                C=C,
                max_iter=max_iter,
                solver="lbfgs",
                class_weight=class_weight,
                multi_class="auto"
            )

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            results.append({
                "C": C,
                "max_iter": max_iter,
                "class_weight": class_weight,
                "accuracy": accuracy_score(y_test, y_pred),
                "macro_f1": f1_score(y_test, y_pred, average="macro"),
                "weighted_f1": f1_score(y_test, y_pred, average="weighted")
            })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by=["macro_f1", "accuracy"],
    ascending=False
).reset_index(drop=True)

results_df

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

,C,max_iter,class_weight,accuracy,macro_f1,weighted_f1
0,1.00,500,None,0.093750,0.060500,0.060500
1,1.00,500,balanced,0.093750,0.060500,0.060500
2,1.00,1000,None,0.093750,0.060500,0.060500
3,1.00,1000,balanced,0.093750,0.060500,0.060500
4,1.00,3000,None,0.093750,0.060500,0.060500
5,1.00,3000,balanced,0.093750,0.060500,0.060500
6,10.00,500,None,0.093750,0.059831,0.059831
7,10.00,500,balanced,0.093750,0.059831,0.059831
8,10.00,1000,None,0.093750,0.059831,0.059831
9,10.00,1000,balanced,0.093750,0.059831,0.059831


In [24]:
best_row = results_df.iloc[0]
best_row

,0
C,1.0
max_iter,500
class_weight,None
accuracy,0.09375
macro_f1,0.0605
weighted_f1,0.0605


train the model with best hyperparameter

In [25]:
best_model = LogisticRegression(
    C=float(best_row["C"]),
    max_iter=int(best_row["max_iter"]),
    solver="lbfgs",
    class_weight=best_row["class_weight"] if pd.notna(best_row["class_weight"]) else None,
    multi_class="auto"
)

best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

print("Best AA model accuracy:", accuracy_score(y_test, best_pred))
print("Best AA model macro F1:", f1_score(y_test, best_pred, average="macro"))
print("Best AA model weighted F1:", f1_score(y_test, best_pred, average="weighted"))

Best AA model accuracy: 0.09375
Best AA model macro F1: 0.060500372023809514
Best AA model weighted F1: 0.060500372023809514


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
